# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login

login()

In [2]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, duckdb, pandas as pd, numpy as np
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"
os.makedirs(BASE, exist_ok=True)

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

Mounted at /content/drive


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# Reload the finalized Week 5 v2 artifacts (Feb-Apr features, May label, april_clicks included)
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"
features = pd.read_csv(f"{BASE}/features_v2.csv")
labels = pd.read_csv(f"{BASE}/labels_v2.csv")
data_model = pd.read_csv(f"{BASE}/data_model_v2.csv")

model_cols = ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
              "february_clicks", "click_through_rate", "weighted_position", "momentum",
              "active_days", "click_through_rate_missing", "weighted_position_missing", "momentum_missing"]

X = data_model[model_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return y_true.iloc[order[:k]].mean()

# --- BEFORE: naive random split (no grouping) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.3, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

auc_random = roc_auc_score(y_test_r.reset_index(drop=True), scores_random)
p20_random = precision_at_k(y_test_r.reset_index(drop=True), scores_random, 20)

print("RANDOM SPLIT (naive):")
print("AUC:", auc_random, "| Precision@20:", p20_random)

RANDOM SPLIT (naive):
AUC: 0.9276913585369413 | Precision@20: 1.0


In [6]:
# --- AFTER: grouped split by client (the honest version) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

auc_grouped = roc_auc_score(y_test_g.reset_index(drop=True), scores_grouped)
p20_grouped = precision_at_k(y_test_g.reset_index(drop=True), scores_grouped, 20)

print("GROUPED SPLIT (by client, honest):")
print("AUC:", auc_grouped, "| Precision@20:", p20_grouped)

print("\nClient overlap in random split (should be > 0, proving the leak path):",
      len(set(data_model.iloc[X_train_r.index]["client_hash_id"]) &
          set(data_model.iloc[X_test_r.index]["client_hash_id"])))

GROUPED SPLIT (by client, honest):
AUC: 0.9302290957526194 | Precision@20: 0.4

Client overlap in random split (should be > 0, proving the leak path): 49


In [7]:
before_after = pd.DataFrame({
    "split": ["Random (naive)", "Grouped by client (honest)"],
    "AUC": [auc_random, auc_grouped],
    "precision@20": [p20_random, p20_grouped]
})
before_after

,split,AUC,precision@20
0,Random (naive),0.927691,1.0
1,Grouped by client (honest),0.930229,0.4


In [8]:
p50_random = precision_at_k(y_test_r.reset_index(drop=True), scores_random, 50)
p50_grouped = precision_at_k(y_test_g.reset_index(drop=True), scores_grouped, 50)

before_after_full = pd.DataFrame({
    "split": ["Random (naive)", "Grouped by client (honest)"],
    "AUC": [auc_random, auc_grouped],
    "precision@20": [p20_random, p20_grouped],
    "precision@50": [p50_random, p50_grouped],
})
before_after_full

,split,AUC,precision@20,precision@50
0,Random (naive),0.927691,1.0,1.00
1,Grouped by client (honest),0.930229,0.4,0.54


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Run the attack checklist against model_cols explicitly, one item at a time

print("--- Leakage Attack Checklist ---\n")

# 1. Timeline check
print("1. Timeline: all features from Feb-Apr, label from May.")
print("   Feature columns:", model_cols)
print("   Label defined from: clicks_april, clicks_may (may only in label, not features)")
print("   april_clicks IS a feature AND shares its reference month with the label's baseline (april) —")
print("   flagged and stress-tested in Week 5 (see train-without test below).\n")

# 2. Train-without test on the suspect feature (april_clicks)
model_cols_no_april = [c for c in model_cols if c not in ["april_clicks", "clicks_window"]]
X_no_april = data_model[model_cols_no_april]
X_train_na, X_test_na = X_no_april.iloc[train_idx], X_no_april.iloc[test_idx]

rf_no_april = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_no_april.fit(X_train_na, y_train_g)
scores_no_april = rf_no_april.predict_proba(X_test_na)[:, 1]
auc_no_april = roc_auc_score(y_test_g.reset_index(drop=True), scores_no_april)

print("2. Train-WITH april_clicks AUC:", auc_grouped)
print("   Train-WITHOUT april_clicks AUC:", auc_no_april)
print(f"   Collapse check: {'MINOR — likely real signal' if auc_grouped - auc_no_april < 0.1 else 'LARGE — investigate further'}\n")

# 3. Product flags check
excluded_flags = ["is_deleted", "is_published", "optimization_flags", "health_score"]
present_flags = [c for c in excluded_flags if c in model_cols]
print("3. Product/decision flags in features:", present_flags, "(should be empty)\n")

# 4. Split grouping
print("4. Split grouped by client_hash_id: YES (see section 2)\n")

# 5. Base rate
print("5. Base rate (test set):", y_test_g.mean(), "\n")

# 6. Feature importance sanity check
importances = pd.Series(rf_grouped.feature_importances_, index=model_cols).sort_values(ascending=False)
print("6. Feature importances:\n", importances)
print(f"   Top feature share: {importances.iloc[0]:.1%} —",
      "suspicious, investigate" if importances.iloc[0] > 0.7 else "no single feature dominates")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.